In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install --upgrade scikit-learn imbalanced-learn

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
# from imblearn.over_sampling import SMOTE  <-- WE DO NOT NEED THIS HERE
import os
import re # For cleaning subject IDs
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


# --- 1. DEFINE FILE PATHS ---
print("--- Block 1: Defining File Paths ---")
output_dir = "/kaggle/working/master_dataset"
os.makedirs(output_dir, exist_ok=True)
base_input_dir = "/kaggle/input/all-feature-without-connectivity"

# (File lists are unchanged)
asd_files = [
    # Entropy Feature
    f"{base_input_dir}/Entropy Feature/ASD_entropy_features.csv",
    # Self_similarity feature
    f"{base_input_dir}/Self_similarity feature/ASD_dfa_band_features.csv",
    f"{base_input_dir}/Self_similarity feature/ASD_hfd_band_features.csv",
    # Spatial_feature
    f"{base_input_dir}/Spatial_feature/ml_ready_faa_features_ASD.csv",
    f"{base_input_dir}/Spatial_feature/ml_ready_gen_features_ASD.csv",
    # Spectral power
    f"{base_input_dir}/Spectral power/ASD_combined_power_features.csv",
    f"{base_input_dir}/Spectral power/ml_ready_centroid_features_ASD.csv",
    f"{base_input_dir}/Spectral power/ml_ready_sef_features_ASD.csv"
]
td_files = [
    # Entropy Feature
    f"{base_input_dir}/Entropy Feature/TD_entropy_features.csv",
    # Self_similarity feature
    f"{base_input_dir}/Self_similarity feature/TD_dfa_band_features.csv",
    f"{base_input_dir}/Self_similarity feature/TD_hfd_band_features.csv",
    # Spatial_feature
    f"{base_input_dir}/Spatial_feature/ml_ready_faa_features_TD.csv",
    f"{base_input_dir}/Spatial_feature/ml_ready_gen_features_TD.csv",
    # Spectral power
    f"{base_input_dir}/Spectral power/TD_combined_power_features.csv",
    f"{base_input_dir}/Spectral power/ml_ready_centroid_features_TD.csv",
    f"{base_input_dir}/Spectral power/ml_ready_sef_features_TD.csv"
]

# --- 2. DEFINE HELPER FUNCTIONS ---
print("--- Block 2: Defining Helper Functions ---")

def extract_subject_number(subject_id):
    subject_id_str = str(subject_id)
    match = re.search(r'\d+', subject_id_str)
    if match:
        return match.group(0).zfill(3)
    return None

def merge_feature_files(file_list, group_name):
    master_df = None
    for file_path in file_list:
        try:
            feature_df = pd.read_csv(file_path)
            
            if feature_df.empty:
                print(f"  > Warning: File is empty: {file_path}. Skipping.")
                continue
            
            feature_df['subject_id'] = feature_df['subject_id'].apply(extract_subject_number)
            feature_df['subject_id'] = feature_df['subject_id'].apply(lambda x: f"{group_name}_{x}")
            
            if master_df is None:
                master_df = feature_df
                if 'label' not in master_df.columns:
                        print(f"  > Warning: Base file {file_path} has no 'label'.")
                else:
                        print(f"  > Using {file_path} as base.")
            else:
                if 'label' in feature_df.columns:
                    feature_df = feature_df.drop(columns=['label'])
                
                feature_df = feature_df.dropna(axis=1, how='all')
                master_df = pd.merge(master_df, feature_df, on="subject_id", how="inner")
                
        except FileNotFoundError:
            print(f"Warning: File not found: {file_path}. Skipping.")
        except Exception as e:
            print(f"Error merging {file_path}: {e}")
            
    return master_df

# --- 3. RUN THE MERGING & SPLITTING PROCESS ---
print("\n--- Block 3: Executing Data Pipeline ---")

print("\nStep 1: Merging ASD features...")
asd_master_df = merge_feature_files(asd_files, "ASD") 
if asd_master_df is not None:
    print(f"  → ASD master shape: {asd_master_df.shape}")
else:
    print("  → Error: ASD master DataFrame is None or empty.")

print("Step 2: Merging TD features...")
td_master_df = merge_feature_files(td_files, "TD") 
if td_master_df is not None:
    print(f"  → TD master shape: {td_master_df.shape}")
else:
    print("  → Error: TD master DataFrame is None or empty.")

if asd_master_df is None or asd_master_df.empty or td_master_df is None or td_master_df.empty:
    print("\nError: One or both master DataFrames are empty. Cannot proceed.")
else:
    master_df = pd.concat([asd_master_df, td_master_df], ignore_index=True)
    
    master_df = master_df.loc[:, ~master_df.columns.str.endswith('_y')]
    master_df.columns = master_df.columns.str.replace('_x', '')
    
    initial_cols = master_df.shape[1]
    master_df = master_df.dropna(axis=1, how='all')
    print(f"  > Dropped {initial_cols - master_df.shape[1]} all-NaN columns.")


    print(f"\nStep 3: Combined Master DataFrame created.")
    print(f"  → Full master shape: {master_df.shape}")
    print(f"  → Class distribution:\n{master_df['label'].value_counts()}")
    
    # --- 4. DEFINE TEST SUBJECTS ---
    print("\nStep 4: Manually selecting test subjects...")
    
    asd_subject_ids = master_df[master_df['label'] == 'ASD']['subject_id'].unique()
    td_subject_ids = master_df[master_df['label'] == 'TD']['subject_id'].unique()
    
    if len(asd_subject_ids) < 4 or len(td_subject_ids) < 4:
        print(f"  → Error: Not enough unique subjects for the split.")
        print(f"  → Found {len(asd_subject_ids)} ASD subjects and {len(td_subject_ids)} TD subjects.")
        print("  → Cannot proceed with manual 4-subject split.")
    else:
        test_asd_subjects = list(asd_subject_ids[:4])
        test_td_subjects = list(td_subject_ids[:4])
        test_subject_list = test_asd_subjects + test_td_subjects
        
        print(f"  → Selected ASD test subjects: {test_asd_subjects}")
        print(f"  → Selected TD test subjects: {test_td_subjects}")

        # --- 5. PERFORM MANUAL SPLIT ---
        print("\nStep 5: Performing manual subject-wise split...")
        
        # This is your clean test set
        test_df = master_df[master_df['subject_id'].isin(test_subject_list)]
        # This is your clean, UNBALANCED training set
        train_df = master_df[~master_df['subject_id'].isin(test_subject_list)]
        
        print(f"  → Split complete.")
        print(f"  → Total unique subjects found: {len(master_df['subject_id'].unique())}")
        print(f"  → Training subjects: {len(train_df['subject_id'].unique())}")
        print(f"  → Testing subjects: {len(test_df['subject_id'].unique())}")
        print(f"  → Train Class Distribution:\n{train_df['label'].value_counts()}")
        print(f"  → Test Class Distribution:\n{test_df['label'].value_counts()}")
        
        
        # --- 6. (OPTIONAL) PREPARE DATA FOR SAVING ---
        # We can just save train_df and test_df directly
        
        
        # --- 7. APPLY SMOTE (TO TRAINING DATA ONLY) ---
        # *** THIS ENTIRE BLOCK IS REMOVED TO PREVENT DATA LEAKAGE ***
        print("\nStep 7: SMOTE application (SKIPPED).")
        print("  → This script will now save the UNBALANCED training data.")
        print("  → SMOTE must be applied *inside* the model training script's pipeline.")

        
        # --- 8. SAVE THE FINAL DATASETS ---
        print("\nStep 8: Saving final datasets...")
        
        # --- 1. ***MODIFIED***: Save the UNBALANCED Training Set ---
        # We are now saving `train_df` from Step 5.
        # It already contains 'subject_id' and 'label', so we can save it directly.
        # We drop subject_id as it's not a feature for the model.
        train_df_to_save = train_df.drop(columns=['subject_id'])
        
        # *** I recommend this new name to be clear ***
        train_path = os.path.join(output_dir, "train_UNBALANCED_dataset.csv") 
        train_df_to_save.to_csv(train_path, index=False)
        print(f"\n  ✅ Saved UNBALANCED training set to: {train_path}")
        
        # --- 2. Unbalanced Test Set (Your original logic was correct) ---
        # test_df from Step 5 already has all columns.
        
        # Reorder columns to have subject_id and label first
        cols = ['subject_id', 'label'] + [col for col in test_df.columns if col not in ['subject_id', 'label']]
        test_unbalanced_df = test_df[cols]
        
        test_path = os.path.join(output_dir, "test_unbalanced_dataset.csv")
        test_unbalanced_df.to_csv(test_path, index=False)
        print(f"  ✅ Saved unbalanced test set to: {test_path}")

print("\n\n🎯 Process complete.")

--- Block 1: Defining File Paths ---
--- Block 2: Defining Helper Functions ---

--- Block 3: Executing Data Pipeline ---

Step 1: Merging ASD features...
  > Using /kaggle/input/all-feature-without-connectivity/Entropy Feature/ASD_entropy_features.csv as base.
  → ASD master shape: (15, 2793)
Step 2: Merging TD features...
  > Using /kaggle/input/all-feature-without-connectivity/Entropy Feature/TD_entropy_features.csv as base.
  → TD master shape: (19, 2793)
  > Dropped 315 all-NaN columns.

Step 3: Combined Master DataFrame created.
  → Full master shape: (34, 2478)
  → Class distribution:
label
TD     19
ASD    15
Name: count, dtype: int64

Step 4: Manually selecting test subjects...
  → Selected ASD test subjects: ['ASD_001', 'ASD_002', 'ASD_003', 'ASD_004']
  → Selected TD test subjects: ['TD_001', 'TD_002', 'TD_003', 'TD_004']

Step 5: Performing manual subject-wise split...
  → Split complete.
  → Total unique subjects found: 34
  → Training subjects: 26
  → Testing subjects: 8


In [ ]:
### =========================================================================
### BLOCK 1: IMPORTS & LOADING (WITH KEYERROR FIX)
### =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import re 

# --- Preprocessing & Pipeline ---
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Classifier Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# --- File paths ---
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv'

print(f"Loading data from {TRAIN_FILE} and {TEST_FILE}...")
n_train = 0 # This will store the original split point

# --- Load Training Data ---
try:
    train_df = pd.read_csv(TRAIN_FILE)
    n_train = len(train_df) # *** FIX: Store the number of rows ***
    y_train_text = train_df['label']
    X_train = train_df.drop(columns=['label'])
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find training file: {TRAIN_FILE}")
    exit()

# --- Load Testing Data ---
try:
    test_df = pd.read_csv(TEST_FILE)
    y_test_text = test_df['label']
    # *** FIX: Drop 'subject_id' here so columns match X_train ***
    X_test = test_df.drop(columns=['label', 'subject_id'], errors='ignore') 
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find testing file: {TEST_FILE}")
    exit()

# --- Encode Labels ---
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train_text)
y_test = encoder.transform(y_test_text)
class_names = encoder.classes_

print(f"Loaded {len(X_train)} train samples and {len(X_test)} test samples.")
print(f"Original feature count: {X_train.shape[1]}")
print(f"Data split point will be at index: {n_train}")
print("-" * 50)


### =========================================================================
### BLOCK 2: DEFINE DOMAIN KNOWLEDGE (BRAIN REGIONS)
### =========================================================================
print("\nDefining brain region mappings...")

# Get a list of all channels from the columns (e.g., 'sampen_alpha_Fp1')
all_columns = X_train.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1:
        all_channels.add(parts[-1]) 
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}

print(f"Found {len(all_channels)} unique channel names in the data.")

# --- Define Standard Regions ---
REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
    'temporal': ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10'],
    'occipital': ['O1', 'Oz', 'O2', 'PO7', 'PO8']
}

# --- Create final region-to-channel mapping ---
final_regions = {}
for region_name, std_channels in REGIONS.items():
    found_channels = [ch for ch in std_channels if ch in all_channels]
    final_regions[region_name] = found_channels
    print(f"  - {region_name.title()}: Found {len(found_channels)} channels {found_channels}")

# Create a 'global' region mapping for all channels found
final_regions['global'] = list(all_channels)
print("-" * 50)


### =========================================================================
### BLOCK 3: DEFINE THE FEATURE AGGREGATION FUNCTION (FIXED)
### =========================================================================
def aggregate_features(df, region_map):
    """
    Takes a wide dataframe (p=2476) and aggregates it down to 
    a small, hypothesis-driven dataframe (p=35).
    *** FIX: This version does NOT use 'subject_id' ***
    """
    print(f"\nAggregating features for {len(df)} subjects...")
    
    # Store new features in a dictionary
    agg_features = {}
    
    # --- 1. Keep PRE-AGGREGATED Features (Asymmetry, GFP) ---
    print("  - Hypothesis 0: Keeping 7 pre-aggregated features...")
    pre_agg_cols = [
        'frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 
        'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma'
    ]
    for col in pre_agg_cols:
        if col in df.columns:
            agg_features[col] = df[col]
        else:
            print(f"    - Warning: Pre-aggregated column '{col}' not found. Filling with 0.")
            agg_features[col] = 0

    # --- 2. Helper function for Complexity/Entropy (e.g., 'hfd_alpha_Fp1') ---
    def get_region_avg(df, feature_prefix, band, region_name):
        channels = region_map.get(region_name, [])
        if not channels:
            return pd.Series(0, index=df.index) # Return a Series of zeros
        
        cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
        
        if not cols_to_avg:
             return pd.Series(0, index=df.index)
        
        return df[cols_to_avg].mean(axis=1)

    # --- 3. Helper function for SEF/Centroid (e.g., 'Fp1_Centroid') ---
    def get_region_avg_spec(df, feature_suffix, region_name):
        channels = region_map.get(region_name, [])
        if not channels:
            return pd.Series(0, index=df.index)
        
        cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
        
        if not cols_to_avg:
             return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    # --- 4. Create NEW Aggregated Features ---
    print("  - Hypothesis 1 & 2: Creating 8 Complexity features (HFD, DFA)...")
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')

    print("  - Hypothesis 3: Creating 2 Entropy features (Permutation)...")
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')

    print("  - Hypothesis 4: Creating 2 Spectral Shape features (SEF, Centroid)...")
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')

    print("  - Hypothesis 5: Creating 16 Spectral Power features (Abs/Rel)...")
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')

    # --- 5. Convert to DataFrame ---
    final_agg_df = pd.DataFrame(agg_features)
    final_agg_df = final_agg_df.fillna(0) # Fill any NaNs from failed averages with 0
    
    print(f"  - Aggregation complete. New shape: {final_agg_df.shape}")
    return final_agg_df
print("-" * 50)


### =========================================================================
### BLOCK 4: APPLY AGGREGATION AND RE-SPLIT DATA (FIXED)
### =========================================================================
print("\nApplying aggregation to combined data...")

# *** FIX: X_train and X_test now have matching columns ***
combined_df = pd.concat([X_train, X_test], ignore_index=True)

# --- Apply the aggregation ---
aggregated_master_df = aggregate_features(combined_df, final_regions)

print("\nRe-splitting into Train and Test sets...")

# *** FIX: Use n_train and .iloc to split the data by index ***
X_train_agg = aggregated_master_df.iloc[:n_train]
X_test_agg = aggregated_master_df.iloc[n_train:]

print(f"New X_train_agg shape: {X_train_agg.shape}")
print(f"New X_test_agg shape: {X_test_agg.shape}")
print(f"\nTotal aggregated features: {len(X_train_agg.columns)}")
print("-" * 50)


### =========================================================================
### BLOCK 5: DEFINE NEW, SIMPLER ML SEARCH SPACE
### =========================================================================
def get_simple_model_search_space():
    """
    Returns a search space for a simple (Scaler + Classifier) pipeline.
    We use class_weight='balanced' to handle the unbalanced data.
    """
    
    # --- Base pipeline (NO PCA, NO SELECTION) ---
    pipeline_simple = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', 'passthrough')
    ])

    print("\nModel search space defined (Simple Scaler + Classifier).")
    
    search_space = {
        
        # --- RandomForest ---
        'RF_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [RandomForestClassifier(random_state=42, class_weight='balanced')],
                'classifier__n_estimators': [50, 100, 200],
                'classifier__max_depth': [2, 3, 5],
                'classifier__min_samples_leaf': [3, 5]
            }
        },
        
        # --- SVM ---
        'SVM_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [SVC(probability=True, random_state=42, class_weight='balanced')],
                'classifier__C': [0.01, 0.1, 1.0, 10.0],
                'classifier__gamma': [0.01, 0.1, 'scale']
            }
        },

        # --- Logistic Regression ---
        'LogReg_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, class_weight='balanced')],
                'classifier__penalty': ['l1', 'l2'],
                'classifier__C': [0.01, 0.1, 1.0, 10.0]
            }
        },
    }
    
    return search_space
print("-" * 50)


### =========================================================================
### BLOCK 6: RUN PIPELINE AND SHOW FINAL, TRUSTWORTHY RESULTS
### =========================================================================

# --- Get the new search space ---
simple_search_space = get_simple_model_search_space()

# --- Use RepeatedStratifiedKFold for stable CV ---
cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
print(f"\nUsing RepeatedStratifiedKFold (5 splits, 5 repeats) on n={len(X_train_agg)} training samples.")

all_results = []

for model_name, config in simple_search_space.items():
    print(f"\n--- Running GridSearch for: {model_name} ---")
    
    grid_search = GridSearchCV(
        estimator=config['pipeline'], 
        param_grid=config['params'], 
        cv=cv_strategy, 
        scoring='accuracy',
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    
    grid_search.fit(X_train_agg, y_train)
    
    print(f"  - Best CV Score: {grid_search.best_score_ * 100:.2f}%")
    all_results.append({
        'model_name': model_name,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    })

# --- Sort and show CV results ---
all_results.sort(key=lambda x: x['best_score'], reverse=True)
print("\n" + "="*40)
print("--- AGGREGATED MODEL CV COMPARISON ---")
print("="*40)
for result in all_results:
    print(f"Model: {result['model_name']}")
    print(f"  Best Mean CV Accuracy: {result['best_score'] * 100:.2f}%")

# --- Analyze the BEST model on the TEST set ---
best_overall_result = all_results[0]
best_pipeline = best_overall_result['best_estimator']

print(f"\n\n--- Final Performance on HELD-OUT TEST DATA ---")
print(f"--- (Winner: {best_overall_result['model_name']}) ---")
# *** MODIFICATION: Added print statement for best hyperparameters ***
print(f"Winning Hyperparameters: {best_overall_result['best_params']}")

y_pred = best_pipeline.predict(X_test_agg)

print(f"Classification Report (on {len(y_test)} test subjects):")
print(classification_report(y_test, y_pred, target_names=class_names))

print("Plotting Confusion Matrix...")
try:
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Test Data Confusion Matrix (Aggregated Features)')
    plt.savefig('confusion_matrix_aggregated.png')
    print("Saved 'confusion_matrix_aggregated.png'")
    plt.close()
except Exception as e:
    print(f"Could not plot confusion matrix: {e}")

print("\n--- Model run complete. ---")


### =========================================================================
### BLOCK 7: ANALYZE FINAL FEATURE IMPORTANCE
### =========================================================================
print("\n" + "="*40)
print("--- FEATURE IMPORTANCE OF WINNING MODEL ---")
print("="*40)

# Get the final fitted classifier from the best pipeline
best_classifier = best_pipeline.named_steps['classifier']

# Get the feature names from our aggregated data
feature_names = X_train_agg.columns.tolist()

importances = None

# Method 1: For RandomForest
if isinstance(best_classifier, RandomForestClassifier):
    print("Model is RandomForest. Using .feature_importances_")
    importances = best_classifier.feature_importances_

# Method 2: For LogisticRegression (L1 or L2)
elif isinstance(best_classifier, LogisticRegression):
    print("Model is LogisticRegression. Using abs(.coef_[0])")
    importances = np.abs(best_classifier.coef_[0])

# Method 3: For SVM
elif isinstance(best_classifier, SVC):
    print("Model is SVM. Built-in importances are not reliable.")
    print("To get feature importance for SVM, you would need to run a")
    print("more complex analysis like SHAP or Permutation Importance.")

# --- Print the results ---
if importances is not None:
    # Create a DataFrame to view the results
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    
    # Sort by importance
    feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
    
    print("\n--- Top 10 Most Important Features ---")
    print(feature_importance_df.head(10))
    
    # Plot the results
    try:
        plt.figure(figsize=(10, 8))
        plt.title("Top 10 Feature Importances (Aggregated Model)")
        plt.barh(
            feature_importance_df['feature'].head(10), 
            feature_importance_df['importance'].head(10)
        )
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.gca().invert_yaxis() # Show best feature at the top
        plt.tight_layout()
        plt.savefig('feature_importance_aggregated.png')
        print("\nSaved 'feature_importance_aggregated.png'")
        plt.close()
    except Exception as e:
        print(f"Could not plot feature importances: {e}")

else:
    print(f"Cannot get simple feature importances for model type: {type(best_classifier)}")

print("\n--- Full analysis complete. ---")

In [2]:
### =========================================================================
### BLOCK 1: IMPORTS & LOADING (FIXED)
### =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import re 
import shap # <-- NEW: Import SHAP

# --- Preprocessing & Pipeline ---
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Classifier Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

print("--- RUNNING FINAL ANALYSIS (v4) ON ALL 34 SUBJECTS ---")
print("--- Adding SHAP analysis for KNN model ---")

# --- 1. Load ALL Data ---
# *** YOU MUST ensure these file names are correct in your environment ***
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv' 

n_train = 0 
try:
    train_df = pd.read_csv(TRAIN_FILE)
    n_train = len(train_df)
    y_train_text = train_df['label']
    # *** FIX: Must drop 'subject_id' here to match X_test ***
    X_train = train_df.drop(columns=['label', 'subject_id'], errors='ignore')
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find training file: {TRAIN_FILE}")
    print("Please ensure 'train_UNBALANCED_dataset.csv' is in the correct path.")
    exit()

try:
    test_df = pd.read_csv(TEST_FILE)
    y_test_text = test_df['label']
    X_test = test_df.drop(columns=['label', 'subject_id'], errors='ignore') 
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find testing file: {TEST_FILE}")
    print("Please ensure 'test_unbalanced_dataset (1).csv' is in the correct path.")
    exit()

# --- Combine ALL labels ---
y_full = np.concatenate([y_train_text, y_test_text])
y_full_encoded = LabelEncoder().fit_transform(y_full)
class_names_full = LabelEncoder().fit(y_full).classes_ # Get class names for SHAP

# --- Define Regions (abbreviated) ---
all_columns = X_train.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1: all_channels.add(parts[-1]) 
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}
REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
}
final_regions = {}
for region_name, std_channels in REGIONS.items():
    final_regions[region_name] = [ch for ch in std_channels if ch in all_channels]
final_regions['global'] = list(all_channels)

### =========================================================================
### BLOCK 3: DEFINE THE FEATURE AGGREGATION FUNCTION (37 FEATURES)
### =========================================================================
def aggregate_features(df, region_map):
    """
    Takes a wide dataframe (p=2476) and aggregates it down to 
    a small, hypothesis-driven dataframe.
    Creates 37 features (35 base + 2 interactions).
    """
    print(f"\nAggregating features for {len(df)} subjects...")
    
    agg_features = {}
    
    # --- 1. Pre-Aggregated Features (7 features) ---
    print("  - Hypothesis 0: Keeping 7 pre-aggregated features...")
    pre_agg_cols = [
        'frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 
        'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma'
    ]
    for col in pre_agg_cols:
        if col in df.columns: agg_features[col] = df[col]
        else: agg_features[col] = 0

    # --- 2. Helper Functions ---
    def get_region_avg(df, feature_prefix, band, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    def get_region_avg_spec(df, feature_suffix, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    # --- 3. Create Aggregated Features (28 features) ---
    print("  - Hypotheses 1-5: Creating 28 aggregated features...")
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')

    # --- 4. Create Interaction Features (2 features) ---
    print("  - Hypothesis 6: Creating 2 new interaction/ratio features...")
    
    # Interaction: Asymmetry * Complexity
    agg_features['asymm_x_dfa'] = agg_features.get('frontal_asymmetry', 0) * agg_features.get('global_dfa_beta', 0)
    
    # Ratio: Delta Power / Beta Power
    numerator = agg_features.get('global_abs_power_delta', 0)
    denominator = agg_features.get('global_abs_power_beta', 0) + 1e-6
    agg_features['delta_beta_ratio'] = numerator / denominator

    # --- 5. Convert to DataFrame ---
    final_agg_df = pd.DataFrame(agg_features).fillna(0)
    
    print(f"  - Aggregation complete. New shape: {final_agg_df.shape}")
    return final_agg_df

### =========================================================================
### BLOCK 4: APPLY AGGREGATION
### =========================================================================
print("\nAggregating all 34 samples...")
# *** FIX: This concat will now work correctly ***
combined_df = pd.concat([X_train, X_test], ignore_index=True)
X_full_agg = aggregate_features(combined_df, final_regions)

print(f"Full aggregated dataset shape: {X_full_agg.shape}")
print(f"Full labels shape: {y_full_encoded.shape}")

### =========================================================================
### BLOCK 5: DEFINE ML SEARCH SPACE
### =========================================================================
def get_simple_model_search_space():
    pipeline_simple = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', 'passthrough')
    ])
    
    print("\nModel search space defined (Simple Scaler + 5 Classifiers).")
    
    search_space = {
        'RF_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [RandomForestClassifier(random_state=42, class_weight='balanced')],
                'classifier__n_estimators': [50, 100], 
                'classifier__max_depth': [2, 3, 5],
            }
        },
        'SVM_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [SVC(probability=True, random_state=42, class_weight='balanced')],
                'classifier__C': [0.1, 1.0, 10.0], 
                'classifier__gamma': [0.01, 0.1, 'scale']
            }
        },
        'LogReg_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, class_weight='balanced')],
                'classifier__penalty': ['l1', 'l2'], 
                'classifier__C': [0.1, 1.0, 10.0]
            }
        },
        'DTree_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [DecisionTreeClassifier(random_state=42, class_weight='balanced')],
                'classifier__max_depth': [2, 3, 4, 5],
                'classifier__min_samples_leaf': [3, 5]
            }
        },
        'KNN_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [KNeighborsClassifier()],
                'classifier__n_neighbors': [3, 5, 7], 
                'classifier__weights': ['uniform', 'distance']
            }
        }
    }
    return search_space

simple_search_space = get_simple_model_search_space()

### =========================================================================
### BLOCK 6: RUN PIPELINE (MODIFIED)
### =========================================================================
cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
print(f"\nUsing RepeatedStratifiedKFold (5 splits, 10 repeats) on all n={len(X_full_agg)} samples.")

all_results = []

for model_name, config in simple_search_space.items():
    print(f"\n--- Running GridSearch for: {model_name} ---")
    grid_search = GridSearchCV(
        estimator=config['pipeline'], 
        param_grid=config['params'], 
        cv=cv_strategy, 
        scoring='accuracy',
        n_jobs=-1,
        refit=True
    )
    
    grid_search.fit(X_full_agg, y_full_encoded)
    
    print(f"  - Best Mean CV Score: {grid_search.best_score_ * 100:.2f}%")
    all_results.append({
        'model_name': model_name,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_, # <-- *** MODIFICATION: Store params ***
        'best_estimator': grid_search.best_estimator_
    })

### =========================================================================
### BLOCK 7: REPORT FINAL RESULTS (MODIFIED)
### =========================================================================
all_results.sort(key=lambda x: x['best_score'], reverse=True)
print("\n" + "="*40)
print("--- FINAL MODEL COMPARISON (v4) ---")
print("="*40)
for result in all_results:
    print(f"Model: {result['model_name']}")
    print(f"  Best Mean CV Accuracy: {result['best_score'] * 100:.2f}%")

# --- Get the winning model for SHAP ---
best_overall_result = all_results[0]
best_pipeline = best_overall_result['best_estimator']

print(f"\nWinning model is: {best_overall_result['model_name']}")
# --- *** MODIFICATION: Print the winning hyperparameters *** ---
print(f"Winning Hyperparameters: {best_overall_result['best_params']}")


### =========================================================================
### BLOCK 8: NEW --- SHAP ANALYSIS FOR WINNING MODEL
### =========================================================================
print("\n" + "="*40)
print("--- SHAP ANALYSIS FOR WINNING MODEL ---")
print("="*40)

try:
    # --- 1. Prepare data and explainer ---
    
    # We must use the scaled data that the model actually sees.
    # We fit the scaler on the full dataset (since it's being used in a CV pipeline)
    # and then transform the data for SHAP.
    scaler = best_pipeline.named_steps['scaler']
    X_full_agg_scaled = scaler.transform(X_full_agg)
    
    # Convert to a DataFrame to keep feature names
    X_full_agg_scaled_df = pd.DataFrame(X_full_agg_scaled, columns=X_full_agg.columns)
    
    # Get the final classifier
    best_classifier = best_pipeline.named_steps['classifier']

    print("Initializing SHAP KernelExplainer...")
    # We use shap.sample to create a "background" dataset
    # k=10 is small but reasonable for n=34
    background_data = shap.sample(X_full_agg_scaled_df, 10, random_state=42)
    
    # Create the explainer
    explainer = shap.KernelExplainer(best_classifier.predict_proba, background_data)

    # --- 2. Calculate SHAP values ---
    print("Calculating SHAP values... (this may take a moment)")
    # We will explain every sample
    shap_values = explainer.shap_values(X_full_agg_scaled_df)
    print("SHAP calculation complete.")

    # --- 3. Plot the results ---
    # shap_values[1] corresponds to the "positive" class (e.g., 'TD')
    # class_names_full[1] will give its name
    
    print(f"Plotting SHAP summary for class: '{class_names_full[1]}'")
    
    shap.summary_plot(
        shap_values[1], 
        X_full_agg_scaled_df, 
        show=False
    )
    
    plt.title(f"SHAP Summary Plot (Importance for '{class_names_full[1]}')")
    plt.tight_layout()
    plt.savefig('shap_summary_plot_knn.png')
    print("\nSaved 'shap_summary_plot_knn.png'")
    plt.close()

except Exception as e:
    print(f"\n---! SHAP ANALYSIS FAILED !---")
    print(f"Error: {e}")
    print("This can happen if the model type is not fully compatible or due to data size.")

print("\n--- Full analysis complete. ---")

--- RUNNING FINAL ANALYSIS (v4) ON ALL 34 SUBJECTS ---
--- Adding SHAP analysis for KNN model ---

Aggregating all 34 samples...

Aggregating features for 34 subjects...
  - Hypothesis 0: Keeping 7 pre-aggregated features...
  - Hypotheses 1-5: Creating 28 aggregated features...
  - Hypothesis 6: Creating 2 new interaction/ratio features...
  - Aggregation complete. New shape: (34, 37)
Full aggregated dataset shape: (34, 37)
Full labels shape: (34,)

Model search space defined (Simple Scaler + 5 Classifiers).

Using RepeatedStratifiedKFold (5 splits, 10 repeats) on all n=34 samples.

--- Running GridSearch for: RF_Simple ---
  - Best Mean CV Score: 67.81%

--- Running GridSearch for: SVM_Simple ---
  - Best Mean CV Score: 68.24%

--- Running GridSearch for: LogReg_Simple ---
  - Best Mean CV Score: 69.52%

--- Running GridSearch for: DTree_Simple ---
  - Best Mean CV Score: 48.76%

--- Running GridSearch for: KNN_Simple ---
  - Best Mean CV Score: 74.19%

--- FINAL MODEL COMPARISON (v4

  0%|          | 0/34 [00:00<?, ?it/s]

SHAP calculation complete.
Plotting SHAP summary for class: 'TD'

Saved 'shap_summary_plot_knn.png'

--- Full analysis complete. ---


In [3]:
### =========================================================================
### BLOCK 1: IMPORTS & LOADING (FIXED)
### =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import re 
import shap # Import SHAP

# --- Preprocessing & Pipeline ---
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Classifier Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

print("--- RUNNING FINAL ANALYSIS (v6) ON ALL 34 SUBJECTS ---")
print("--- Testing 3 new literature-backed ratio features (Total: 38) ---")

# --- 1. Load ALL Data ---
# *** YOU MUST ensure these file names are correct in your environment ***
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv' 

n_train = 0 
try:
    train_df = pd.read_csv(TRAIN_FILE)
    n_train = len(train_df)
    y_train_text = train_df['label']
    # *** FIX: Must drop 'subject_id' here to match X_test ***
    X_train = train_df.drop(columns=['label', 'subject_id'], errors='ignore')
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find training file: {TRAIN_FILE}")
    raise SystemExit() # This will reliably stop the script

try:
    test_df = pd.read_csv(TEST_FILE)
    y_test_text = test_df['label']
    X_test = test_df.drop(columns=['label', 'subject_id'], errors='ignore') 
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find testing file: {TEST_FILE}")
    raise SystemExit() # This will reliably stop the script

# --- Combine ALL labels ---
y_full = np.concatenate([y_train_text, y_test_text])
y_full_encoded = LabelEncoder().fit_transform(y_full)
class_names_full = LabelEncoder().fit(y_full).classes_ # Get class names for SHAP

# --- Define Regions (abbreviated) ---
all_columns = X_train.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1: all_channels.add(parts[-1]) 
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}
REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
}
final_regions = {}
for region_name, std_channels in REGIONS.items():
    final_regions[region_name] = [ch for ch in std_channels if ch in all_channels]
final_regions['global'] = list(all_channels)

### =========================================================================
### BLOCK 3: DEFINE THE FEATURE AGGREGATION FUNCTION (38 FEATURES)
### =========================================================================
def aggregate_features(df, region_map):
    """
    Takes a wide dataframe (p=2476) and aggregates it down to 
    a small, hypothesis-driven dataframe.
    *** UPDATED: Creates 38 features (35 base + 3 lit-backed ratios) ***
    """
    print(f"\nAggregating features for {len(df)} subjects...")
    
    agg_features = {}
    
    # --- 1. Pre-Aggregated Features (7 features) ---
    print("  - Hypothesis 0: Keeping 7 pre-aggregated features...")
    pre_agg_cols = [
        'frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 
        'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma'
    ]
    for col in pre_agg_cols:
        if col in df.columns: agg_features[col] = df[col]
        else: agg_features[col] = 0

    # --- 2. Helper Functions ---
    def get_region_avg(df, feature_prefix, band, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    def get_region_avg_spec(df, feature_suffix, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    # --- 3. Create Aggregated Features (28 features) ---
    print("  - Hypotheses 1-5: Creating 28 aggregated features...")
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')

    # --- 4. *** NEW *** Create Literature-Backed Ratios (3 features) ---
    print("  - Hypothesis 6-8: Creating 3 new literature-backed ratio features...")
    
    # We must .fillna(0) to prevent errors during multiplication/division
    # A small constant (1e-6) is added to prevent divide-by-zero errors.
    
    # H6 (NEW): Theta/Beta Ratio (TBR)
    numerator_tb = agg_features.get('global_abs_power_theta', 0)
    denominator_tb = agg_features.get('global_abs_power_beta', 0) + 1e-6
    agg_features['global_tbr'] = numerator_tb / denominator_tb

    # H7 (NEW): Delta/Alpha Ratio (DAR)
    numerator_da = agg_features.get('global_abs_power_delta', 0)
    denominator_da = agg_features.get('global_abs_power_alpha', 0) + 1e-6
    agg_features['global_dar'] = numerator_da / denominator_da

    # H8 (NEW): Frontal/Parietal Alpha Ratio
    numerator_fp = agg_features.get('frontal_abs_power_alpha', 0)
    # We need to calculate parietal alpha power first
    agg_features['parietal_abs_power_alpha'] = get_region_avg(df, 'abs_power', 'alpha', 'parietal')
    denominator_fp = agg_features.get('parietal_abs_power_alpha', 0) + 1e-6
    agg_features['fp_alpha_ratio'] = numerator_fp / denominator_fp

    # --- 5. Convert to DataFrame ---
    final_agg_df = pd.DataFrame(agg_features).fillna(0)
    
    print(f"  - Aggregation complete. New shape: {final_agg_df.shape}")
    return final_agg_df

### =========================================================================
### BLOCK 4: APPLY AGGREGATION
### =========================================================================
print("\nAggregating all 34 samples...")
# *** FIX: This concat will now work correctly ***
combined_df = pd.concat([X_train, X_test], ignore_index=True)
X_full_agg = aggregate_features(combined_df, final_regions)

print(f"Full aggregated dataset shape: {X_full_agg.shape}")
print(f"Full labels shape: {y_full_encoded.shape}")

### =========================================================================
### BLOCK 5: DEFINE ML SEARCH SPACE
### =========================================================================
def get_simple_model_search_space():
    pipeline_simple = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', 'passthrough')
    ])
    
    print("\nModel search space defined (Simple Scaler + 5 Classifiers).")
    
    search_space = {
        'RF_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [RandomForestClassifier(random_state=42, class_weight='balanced')],
                'classifier__n_estimators': [50, 100], 
                'classifier__max_depth': [2, 3, 5],
            }
        },
        'SVM_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [SVC(probability=True, random_state=42, class_weight='balanced')],
                'classifier__C': [0.1, 1.0, 10.0], 
                'classifier__gamma': [0.01, 0.1, 'scale']
            }
        },
        'LogReg_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, class_weight='balanced')],
                'classifier__penalty': ['l1', 'l2'], 
                'classifier__C': [0.1, 1.0, 10.0]
            }
        },
        'DTree_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [DecisionTreeClassifier(random_state=42, class_weight='balanced')],
                'classifier__max_depth': [2, 3, 4, 5],
                'classifier__min_samples_leaf': [3, 5]
            }
        },
        'KNN_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [KNeighborsClassifier()],
                'classifier__n_neighbors': [3, 5, 7], 
                'classifier__weights': ['uniform', 'distance']
            }
        }
    }
    return search_space

simple_search_space = get_simple_model_search_space()

### =========================================================================
### BLOCK 6: RUN PIPELINE (MODIFIED)
### =========================================================================
cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
print(f"\nUsing RepeatedStratifiedKFold (5 splits, 10 repeats) on all n={len(X_full_agg)} samples.")

all_results = []

for model_name, config in simple_search_space.items():
    print(f"\n--- Running GridSearch for: {model_name} ---")
    grid_search = GridSearchCV(
        estimator=config['pipeline'], 
        param_grid=config['params'], 
        cv=cv_strategy, 
        scoring='accuracy',
        n_jobs=-1,
        refit=True
    )
    
    grid_search.fit(X_full_agg, y_full_encoded)
    
    print(f"  - Best Mean CV Score: {grid_search.best_score_ * 100:.2f}%")
    all_results.append({
        'model_name': model_name,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_, # <-- *** MODIFICATION: Store params ***
        'best_estimator': grid_search.best_estimator_
    })

### =========================================================================
### BLOCK 7: REPORT FINAL RESULTS (MODIFIED)
### =========================================================================
all_results.sort(key=lambda x: x['best_score'], reverse=True)
print("\n" + "="*40)
print("--- FINAL MODEL COMPARISON (v6) ---")
print("="*40)
for result in all_results:
    print(f"Model: {result['model_name']}")
    print(f"  Best Mean CV Accuracy: {result['best_score'] * 100:.2f}%")

# --- Get the winning model for SHAP ---
best_overall_result = all_results[0]
best_pipeline = best_overall_result['best_estimator']

print(f"\nWinning model is: {best_overall_result['model_name']}")
# --- *** MODIFICATION: Print the winning hyperparameters *** ---
print(f"Winning Hyperparameters: {best_overall_result['best_params']}")


### =========================================================================
### BLOCK 8: SHAP ANALYSIS FOR WINNING MODEL
### =========================================================================
print("\n" + "="*40)
print("--- SHAP ANALYSIS FOR WINNING MODEL ---")
print("="*40)

try:
    # --- 1. Prepare data and explainer ---
    scaler = best_pipeline.named_steps['scaler']
    X_full_agg_scaled = scaler.transform(X_full_agg)
    
    # Convert to a DataFrame to keep feature names
    X_full_agg_scaled_df = pd.DataFrame(X_full_agg_scaled, columns=X_full_agg.columns)
    
    # Get the final classifier
    best_classifier = best_pipeline.named_steps['classifier']

    print("Initializing SHAP KernelExplainer...")
    # We use shap.sample to create a "background" dataset
    background_data = shap.sample(X_full_agg_scaled_df, 10, random_state=42)
    
    # Create the explainer
    explainer = shap.KernelExplainer(best_classifier.predict_proba, background_data)

    # --- 2. Calculate SHAP values ---
    print("Calculating SHAP values... (this may take a moment)")
    shap_values = explainer.shap_values(X_full_agg_scaled_df) # Explain all samples
    print("SHAP calculation complete.")

    # --- 3. Plot the results ---
    print(f"Plotting SHAP summary for class: '{class_names_full[1]}'")
    
    shap.summary_plot(
        shap_values[1], # Use shap values for the positive class
        X_full_agg_scaled_df, 
        show=False
    )
    
    plt.title(f"SHAP Summary Plot (Importance for '{class_names_full[1]}')")
    plt.tight_layout()
    plt.savefig('shap_summary_plot_v6.png')
    print("\nSaved 'shap_summary_plot_v6.png'")
    plt.close()

except Exception as e:
    print(f"\n---! SHAP ANALYSIS FAILED !---")
    print(f"Error: {e}")

print("\n--- Full analysis complete. ---")

--- RUNNING FINAL ANALYSIS (v6) ON ALL 34 SUBJECTS ---
--- Testing 3 new literature-backed ratio features (Total: 38) ---

Aggregating all 34 samples...

Aggregating features for 34 subjects...
  - Hypothesis 0: Keeping 7 pre-aggregated features...
  - Hypotheses 1-5: Creating 28 aggregated features...
  - Hypothesis 6-8: Creating 3 new literature-backed ratio features...
  - Aggregation complete. New shape: (34, 39)
Full aggregated dataset shape: (34, 39)
Full labels shape: (34,)

Model search space defined (Simple Scaler + 5 Classifiers).

Using RepeatedStratifiedKFold (5 splits, 10 repeats) on all n=34 samples.

--- Running GridSearch for: RF_Simple ---
  - Best Mean CV Score: 65.90%

--- Running GridSearch for: SVM_Simple ---
  - Best Mean CV Score: 67.48%

--- Running GridSearch for: LogReg_Simple ---
  - Best Mean CV Score: 68.48%

--- Running GridSearch for: DTree_Simple ---
  - Best Mean CV Score: 48.10%

--- Running GridSearch for: KNN_Simple ---
  - Best Mean CV Score: 71.57%

  0%|          | 0/34 [00:00<?, ?it/s]

SHAP calculation complete.
Plotting SHAP summary for class: 'TD'

Saved 'shap_summary_plot_v6.png'

--- Full analysis complete. ---


In [4]:
### =========================================================================
### FINAL EXPERIMENT (v21) - Corrected Syntax & Hyperparameter Print
### =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import warnings
import re 
import shap

# --- Preprocessing & Pipeline ---
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Classifier Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

print("--- RUNNING FINAL EXPERIMENT (v21) ON ALL 34 SUBJECTS ---")
print("--- Merging v6 Aggregated Features with v19 PAC Features ---")

# =========================================================================
# BLOCK 1: LOAD ALL DATA SOURCES
# =========================================================================

# --- 1. Load Original Wide Data (for v6 Aggregation) ---
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv' 

n_train = 0 
try:
    train_df = pd.read_csv(TRAIN_FILE)
    n_train = len(train_df)
    y_train_text = train_df['label']
    # *** FIX: Must drop 'subject_id' here to match X_test ***
    X_train = train_df.drop(columns=['label', 'subject_id'], errors='ignore')
except FileNotFoundError as e:
    print(f"FATAL ERROR: Cannot find training file: {TRAIN_FILE}")
    raise FileNotFoundError from e

try:
    test_df = pd.read_csv(TEST_FILE)
    y_test_text = test_df['label']
    X_test = test_df.drop(columns=['label', 'subject_id'], errors='ignore') 
except FileNotFoundError as e:
    print(f"FATAL ERROR: Cannot find testing file: {TEST_FILE}")
    raise FileNotFoundError from e

# --- 2. Create Full Label Set (y_full_encoded) ---
y_full = np.concatenate([y_train_text, y_test_text])
y_full_encoded = LabelEncoder().fit_transform(y_full)
class_names_full = LabelEncoder().fit(y_full).classes_

# --- 3. Load New PAC Data ---
PAC_ASD_FILE ='/kaggle/input/pac-dataset/pac_features_ASD.csv'
PAC_TD_FILE ='/kaggle/input/pac-dataset/pac_features_TD.csv'

try:
    pac_df_asd = pd.read_csv(PAC_ASD_FILE)
    pac_df_td = pd.read_csv(PAC_TD_FILE)
    
    X_pac_agg = pd.concat([pac_df_asd, pac_df_td], ignore_index=True)
    X_pac_agg = X_pac_agg.drop(columns=['subject_id', 'label'])
    
    print(f"Successfully loaded {len(X_pac_agg)} PAC feature vectors.")
    
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find PAC CSV files at {PAC_ASD_FILE} or {PAC_TD_FILE}")
    print(f"Error: {e}")
    raise FileNotFoundError from e

# =========================================================================
# BLOCK 2: DEFINE REGIONS (CORRECTED)
# =========================================================================
print("\nDefining brain region mappings (Full 5 Regions)...")

all_columns = X_train.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1: all_channels.add(parts[-1]) 
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}

REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
    'temporal': ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10'],
    'occipital': ['O1', 'Oz', 'O2', 'PO7', 'PO8']
}

final_regions = {}
for region_name, std_channels in REGIONS.items():
    found_channels = [ch for ch in std_channels if ch in all_channels]
    final_regions[region_name] = found_channels
    print(f"  - {region_name.title()}: Found {len(found_channels)} channels")

final_regions['global'] = list(all_channels)
print(f"  - Global: Found {len(final_regions['global'])} channels")


# =========================================================================
# BLOCK 3: DEFINE v6 AGGREGATION FUNCTION (38 FEATURES)
# =========================================================================
def aggregate_v6_features(df, region_map):
    """
    This is the v6 aggregation function.
    Creates 38 features (35 base + 3 lit-backed ratios).
    """
    print(f"\nAggregating v6 features for {len(df)} subjects...")
    agg_features = {}
    pre_agg_cols = ['frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma']
    for col in pre_agg_cols:
        if col in df.columns: agg_features[col] = df[col]
        else: agg_features[col] = 0

    def get_region_avg(df, feature_prefix, band, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    def get_region_avg_spec(df, feature_suffix, region_name):
        channels = region_map.get(region_name, [])
        if not channels: return pd.Series(0, index=df.index)
        cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
        if not cols_to_avg: return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    # Base 28 features
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')

    # v6 Ratios (3 features)
    numerator_tb = agg_features.get('global_abs_power_theta', 0)
    denominator_tb = agg_features.get('global_abs_power_beta', 0) + 1e-6
    agg_features['global_tbr'] = numerator_tb / denominator_tb

    numerator_da = agg_features.get('global_abs_power_delta', 0)
    denominator_da = agg_features.get('global_abs_power_alpha', 0) + 1e-6
    agg_features['global_dar'] = numerator_da / denominator_da
    
    agg_features['parietal_abs_power_alpha'] = get_region_avg(df, 'abs_power', 'alpha', 'parietal')
    numerator_fp = agg_features.get('frontal_abs_power_alpha', 0)
    denominator_fp = agg_features.get('parietal_abs_power_alpha', 0) + 1e-6
    agg_features['fp_alpha_ratio'] = numerator_fp / denominator_fp

    final_agg_df = pd.DataFrame(agg_features).fillna(0)
    print(f"  - v6 Aggregation complete. Shape: {final_agg_df.shape}")
    return final_agg_df

# =========================================================================
# BLOCK 4: CREATE FINAL v21 DATASET
# =========================================================================
print("\nCreating v6 and v21 datasets...")

# 1. Create the v6 (38-feature) dataset
combined_df = pd.concat([X_train, X_test], ignore_index=True)
X_agg_v6 = aggregate_v6_features(combined_df, final_regions)

# 2. Create the v21 (62-feature) dataset
X_agg_v21 = pd.concat([X_agg_v6, X_pac_agg], axis=1)

# --- Sanity Check ---
if X_agg_v6.shape[0] != X_pac_agg.shape[0]:
    print("FATAL ERROR: Mismatch in number of subjects.")
    print(f"v6 aggregated data has {X_agg_v6.shape[0]} rows.")
    print(f"PAC aggregated data has {X_pac_agg.shape[0]} rows.")
    raise SystemExit()

print(f"\n--- v21 Dataset Created Successfully ---")
print(f"v6 baseline features:  {X_agg_v6.shape[1]}")
print(f"v19 PAC features:      {X_pac_agg.shape[1]}")
print(f"v21 combined features: {X_agg_v21.shape[1]}")
print(f"Total subjects:        {X_agg_v21.shape[0]}")


# =========================================================================
# BLOCK 5: DEFINE ML SEARCH SPACE
# =========================================================================
def get_simple_model_search_space():
    pipeline_simple = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', 'passthrough')
    ])
    
    print("\nModel search space defined (Simple Scaler + 5 Classifiers).")
    
    search_space = {
        'RF_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [RandomForestClassifier(random_state=42, class_weight='balanced')],
                'classifier__n_estimators': [50, 100], 'classifier__max_depth': [2, 3, 5],
            }
        },
        'SVM_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [SVC(probability=True, random_state=42, class_weight='balanced')],
                'classifier__C': [0.1, 1.0, 10.0], 'classifier__gamma': [0.01, 0.1, 'scale']
            }
        },
        'LogReg_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, class_weight='balanced')],
                'classifier__penalty': ['l1', 'l2'], 'classifier__C': [0.1, 1.0, 10.0]
            }
        },
        'DTree_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [DecisionTreeClassifier(random_state=42, class_weight='balanced')],
                'classifier__max_depth': [2, 3, 4, 5], 'classifier__min_samples_leaf': [3, 5]
            }
        },
        'KNN_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [KNeighborsClassifier()],
                'classifier__n_neighbors': [3, 5, 7], 'classifier__weights': ['uniform', 'distance']
            }
        }
    }
    return search_space

simple_search_space = get_simple_model_search_space()

# =========================================================================
# BLOCK 6: RUN PIPELINE (MODIFIED)
# =========================================================================
cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
print(f"\nUsing RepeatedStratifiedKFold (5 splits, 10 repeats) on all n=34 samples.")

all_results = []

for model_name, config in simple_search_space.items():
    print(f"\n--- Running GridSearch for: {model_name} ---")
    grid_search = GridSearchCV(
        estimator=config['pipeline'], 
        param_grid=config['params'], 
        cv=cv_strategy, 
        scoring='accuracy',
        n_jobs=-1,
        refit=True
    )
    
    # We fit on the new v21 dataset
    grid_search.fit(X_agg_v21, y_full_encoded)
    
    print(f"  - Best Mean CV Score: {grid_search.best_score_ * 100:.2f}%")
    
    # *** MODIFICATION: Added 'best_params' to the results dictionary ***
    all_results.append({
        'model_name': model_name,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_, # <-- SAVING PARAMS
        'best_estimator': grid_search.best_estimator_
    })

# =========================================================================
# BLOCK 7: REPORT FINAL RESULTS (MODIFIED)
# =========================================================================
all_results.sort(key=lambda x: x['best_score'], reverse=True)
print("\n" + "="*40)
print("--- FINAL MODEL COMPARISON (v21 vs v6) ---")
print("="*40)
print("--- (v6 Baseline Accuracy: ~74.19%) ---")
for result in all_results:
    print(f"Model: {result['model_name']}")
    print(f"  Best Mean CV Accuracy: {result['best_score'] * 100:.2f}%")

# --- Get the winning model for SHAP ---
best_overall_result = all_results[0]
best_pipeline = best_overall_result['best_estimator']

print(f"\nWinning model is: {best_overall_result['model_name']}")
# *** MODIFICATION: Added print statement for winning hyperparameters ***
print(f"Winning Hyperparameters: {best_overall_result['best_params']}")


# =========================================================================
# BLOCK 8: SHAP ANALYSIS FOR WINNING MODEL
# =========================================================================
print("\n" + "="*40)
print("--- SHAP ANALYSIS FOR WINNING v21 MODEL ---")
print("="*40)

try:
    # --- 1. Prepare data and explainer ---
    scaler = best_pipeline.named_steps['scaler']
    X_full_agg_scaled = scaler.transform(X_agg_v21)
    
    # Convert to a DataFrame to keep feature names
    X_full_agg_scaled_df = pd.DataFrame(X_full_agg_scaled, columns=X_agg_v21.columns)
    
    # Get the final classifier
    best_classifier = best_pipeline.named_steps['classifier']

    print("Initializing SHAP KernelExplainer...")
    background_data = shap.sample(X_full_agg_scaled_df, 10, random_state=42)
    
    explainer = shap.KernelExplainer(best_classifier.predict_proba, background_data)

    # --- 2. Calculate SHAP values ---
    print("Calculating SHAP values... (this may take a moment)")
    shap_values = explainer.shap_values(X_full_agg_scaled_df)
    print("SHAP calculation complete.")

    # --- 3. Plot the results ---
    print(f"Plotting SHAP summary for class: '{class_names_full[1]}'")
    
    shap.summary_plot(
        shap_values[1], # Use shap values for the positive class
        X_full_agg_scaled_df, 
        show=False,
        max_display=20 # Show top 20 features
    )
    
    plt.title(f"SHAP Summary Plot (v21 Model - Importance for '{class_names_full[1]}')")
    plt.tight_layout()
    plt.savefig('shap_summary_plot_v21.png')
    print("\nSaved 'shap_summary_plot_v21.png'")
    plt.close()

except Exception as e:
    print(f"\n---! SHAP ANALYSIS FAILED !---")
    print(f"Error: {e}")

print("\n--- Full analysis complete. ---")

--- RUNNING FINAL EXPERIMENT (v21) ON ALL 34 SUBJECTS ---
--- Merging v6 Aggregated Features with v19 PAC Features ---
Successfully loaded 34 PAC feature vectors.

Defining brain region mappings (Full 5 Regions)...
  - Frontal: Found 9 channels
  - Central: Found 7 channels
  - Parietal: Found 11 channels
  - Temporal: Found 6 channels
  - Occipital: Found 5 channels
  - Global: Found 85 channels

Creating v6 and v21 datasets...

Aggregating v6 features for 34 subjects...
  - v6 Aggregation complete. Shape: (34, 39)

--- v21 Dataset Created Successfully ---
v6 baseline features:  39
v19 PAC features:      24
v21 combined features: 63
Total subjects:        34

Model search space defined (Simple Scaler + 5 Classifiers).

Using RepeatedStratifiedKFold (5 splits, 10 repeats) on all n=34 samples.

--- Running GridSearch for: RF_Simple ---
  - Best Mean CV Score: 65.29%

--- Running GridSearch for: SVM_Simple ---
  - Best Mean CV Score: 68.14%

--- Running GridSearch for: LogReg_Simple ---


  0%|          | 0/34 [00:00<?, ?it/s]

SHAP calculation complete.
Plotting SHAP summary for class: 'TD'

Saved 'shap_summary_plot_v21.png'

--- Full analysis complete. ---


In [5]:
import numpy as np
import pandas as pd
import warnings
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import bootstrap, wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# =========================================================================
# BLOCK 1: DATA LOADING
# =========================================================================
print("Loading all raw data sources...")

# --- File Paths ---
# !!! YOU MUST UPDATE THESE PATHS !!!
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv'
PAC_ASD_FILE = '/kaggle/input/pac-dataset/pac_features_ASD.csv'
PAC_TD_FILE = '/kaggle/input/pac-dataset/pac_features_TD.csv'

# --- 1. Load Wide Data (X) ---
try:
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)
    
    # Get labels before dropping
    y_train_text = train_df['label']
    y_test_text = test_df['label']

    # Drop labels/subject_id to align columns
    X_train_wide = train_df.drop(columns=['label', 'subject_id'], errors='ignore')
    X_test_wide = test_df.drop(columns=['label', 'subject_id'], errors='ignore')
    
    # Combine into one n=34 wide dataframe
    X_wide_full = pd.concat([X_train_wide, X_test_wide], ignore_index=True)
    
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find wide data files.")
    print(e)
    exit()

# --- 2. Load Labels (y) ---
y_full_text = np.concatenate([y_train_text, y_test_text])
encoder = LabelEncoder()
y = encoder.fit_transform(y_full_text)
print(f"Loaded X data: {X_wide_full.shape}")
print(f"Loaded y labels: {y.shape}")

# --- 3. Load PAC Data (for v21) ---
try:
    pac_df_asd = pd.read_csv(PAC_ASD_FILE)
    pac_df_td = pd.read_csv(PAC_TD_FILE)
    
    X_pac_agg = pd.concat([pac_df_asd, pac_df_td], ignore_index=True)
    # Drop metadata columns
    X_pac_agg = X_pac_agg.drop(columns=['subject_id', 'label'], errors='ignore')
    print(f"Loaded PAC data: {X_pac_agg.shape}")
    
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find PAC data files.")
    print(e)
    exit()

if len(X_wide_full) != len(X_pac_agg):
    print("FATAL ERROR: Row mismatch between wide data and PAC data.")
    exit()

# =========================================================================
# BLOCK 2: AGGREGATION FUNCTIONS & HELPERS
# =========================================================================
print("\nDefining aggregation functions...")

# --- 2a. Define Regions ---
all_columns = X_wide_full.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1: all_channels.add(parts[-1])
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}

REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
    'temporal': ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10'],
    'occipital': ['O1', 'Oz', 'O2', 'PO7', 'PO8']
}
final_regions = {}
for region_name, std_channels in REGIONS.items():
    final_regions[region_name] = [ch for ch in std_channels if ch in all_channels]
final_regions['global'] = list(all_channels)

# --- 2b. Helper Functions ---
def get_region_avg(df, feature_prefix, band, region_name):
    channels = final_regions.get(region_name, [])
    if not channels: return pd.Series(0, index=df.index)
    cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
    if not cols_to_avg: return pd.Series(0, index=df.index)
    return df[cols_to_avg].mean(axis=1)

def get_region_avg_spec(df, feature_suffix, region_name):
    channels = final_regions.get(region_name, [])
    if not channels: return pd.Series(0, index=df.index)
    cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
    if not cols_to_avg: return pd.Series(0, index=df.index)
    return df[cols_to_avg].mean(axis=1)

# --- 2c. Base 35-Feature Aggregation (v-Linear) ---
def build_agg_features(df):
    agg_features = {}
    # 1. Pre-aggregated (7)
    pre_agg_cols = ['frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma']
    for col in pre_agg_cols:
        if col in df.columns: agg_features[col] = df[col]
        else: agg_features[col] = 0
    # 2. Complexity (8)
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')
    # 3. Entropy (2)
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')
    # 4. Spectral Shape (2)
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')
    # 5. Spectral Power (16)
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')
    
    return agg_features

# =========================================================================
# BLOCK 3: CREATE FINAL (n=34) DATASETS
# =========================================================================
print("Building final n=34 datasets...")

# --- 3a. Get Base 35 Features ---
agg_dict_base = build_agg_features(X_wide_full)
X_linear = pd.DataFrame(agg_dict_base).fillna(0)

# --- 3b. Create v4 (37 features) ---
agg_dict_v4 = agg_dict_base.copy()
agg_dict_v4['asymm_x_dfa'] = agg_dict_v4.get('frontal_asymmetry', 0) * agg_dict_v4.get('global_dfa_beta', 0)
num = agg_dict_v4.get('global_abs_power_delta', 0)
den = agg_dict_v4.get('global_abs_power_beta', 0) + 1e-6
agg_dict_v4['delta_beta_ratio'] = num / den
X_v4 = pd.DataFrame(agg_dict_v4).fillna(0)

# --- 3c. Create v6 (38 features) ---
agg_dict_v6 = agg_dict_base.copy()
num_tb = agg_dict_v6.get('global_abs_power_theta', 0)
den_tb = agg_dict_v6.get('global_abs_power_beta', 0) + 1e-6
agg_dict_v6['global_tbr'] = num_tb / den_tb
num_da = agg_dict_v6.get('global_abs_power_delta', 0)
den_da = agg_dict_v6.get('global_abs_power_alpha', 0) + 1e-6
agg_dict_v6['global_dar'] = num_da / den_da
agg_dict_v6['parietal_abs_power_alpha'] = get_region_avg(X_wide_full, 'abs_power', 'alpha', 'parietal')
num_fp = agg_dict_v6.get('frontal_abs_power_alpha', 0)
den_fp = agg_dict_v6.get('parietal_abs_power_alpha', 0) + 1e-6
agg_dict_v6['fp_alpha_ratio'] = num_fp / den_fp
X_v6 = pd.DataFrame(agg_dict_v6).fillna(0)
# Drop the helper column
if 'parietal_abs_power_alpha' in X_v6.columns:
    X_v6 = X_v6.drop(columns=['parietal_abs_power_alpha'])

# --- 3d. Create v21 (v6 + PAC) ---
X_v21 = pd.concat([X_v6, X_pac_agg], axis=1).fillna(0)

print(f"  X_linear shape: {X_linear.shape}")
print(f"  X_v4 shape:     {X_v4.shape}")
print(f"  X_v6 shape:     {X_v6.shape}")
print(f"  X_v21 shape:    {X_v21.shape}")

# =========================================================================
# BLOCK 4: DEFINE FINAL, FIXED MODELS
# =========================================================================
print("\nDefining final, fixed-parameter models...")

# --- model_linear (v-Linear) ---
# Winner: LogReg_Simple
# Params: {'classifier__C': 0.1, 'classifier__penalty': 'l2'} 
model_linear = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', LogisticRegression(
        C=0.1,
        penalty='l2',
        solver='liblinear', # 'l2' penalty can use 'liblinear'
        class_weight='balanced', # From your original code
        random_state=42
    ))
])

# --- model_v4 (Invented) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 7, 'classifier__weights': 'uniform'} 
model_v4 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=7,
        weights='uniform',
        n_jobs=-1
    ))
])

# --- model_v6 (Literature) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 5, 'classifier__weights': 'uniform'} 
model_v6 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=5,
        weights='uniform',
        n_jobs=-1
    ))
])

# --- model_v21 (PAC) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 7, 'classifier__weights': 'uniform'} 
model_v21 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=7,
        weights='uniform',
        n_jobs=-1
    ))
])

# =========================================================================
# BLOCK 5: RUN STATISTICAL EVALUATION
# =========================================================================

# --- 5a. Run Repeated Cross-Validation ---
# We use n_splits=5, n_repeats=10 to match your paper's code 
# Using n_splits=10 is also fine, but we will stick to your method.
# Using 10 repeats to get 50 scores.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

print(f"\nRunning {cv.get_n_splits()} scores (5 splits, 10 repeats) on all n=34 subjects...")
all_scores = {}
all_scores['v_linear'] = cross_val_score(model_linear, X_linear, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v4'] = cross_val_score(model_v4, X_v4, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v6'] = cross_val_score(model_v6, X_v6, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v21'] = cross_val_score(model_v21, X_v21, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("Cross-validation complete.\n")


# --- 5b. Calculate Bootstrap CIs ---
print("Calculating Bootstrap 95% Confidence Intervals...")
results = {}

for model_name, scores in all_scores.items():
    mean_accuracy = np.mean(scores)
    # (data,) must be a tuple for scipy.stats.bootstrap
    ci_result = bootstrap((scores,), np.mean, confidence_level=0.95, 
                          random_state=42, method='BCa')
    
    results[model_name] = {
        'mean': mean_accuracy,
        'ci_low': ci_result.confidence_interval.low,
        'ci_high': ci_result.confidence_interval.high
    }

# --- 5c. Run Paired Hypothesis Tests (Wilcoxon) ---
print("Running paired hypothesis tests (Wilcoxon)...")

# One-sided test: is the model *greater* than the baseline?
p_v4 = wilcoxon(all_scores['v4'], all_scores['v_linear'], alternative='greater').pvalue
p_v6 = wilcoxon(all_scores['v6'], all_scores['v_linear'], alternative='greater').pvalue
p_v21 = wilcoxon(all_scores['v21'], all_scores['v_linear'], alternative='greater').pvalue

raw_p_values = [p_v4, p_v6, p_v21]

# --- 5d. Apply Bonferroni-Holm Correction ---
print("Applying Bonferroni-Holm correction...\n")

reject, p_adjusted, _, _ = multipletests(raw_p_values, alpha=0.05, 
                                         method='holm')

# Store adjusted p-values
results['v4']['p_adj'] = p_adjusted[0]
results['v6']['p_adj'] = p_adjusted[1]
results['v21']['p_adj'] = p_adjusted[2]
results['v_linear']['p_adj'] = np.nan # Baseline has no p-value

# --- 5e. Print Final Results Table ---
print("--- FINAL STATISTICAL RESULTS (n=34) ---")

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.rename_axis('Model').reset_index()
results_df['95% CI'] = results_df.apply(lambda row: f"[{row['ci_low']*100:.2f}% - {row['ci_high']*100:.2f}%]", axis=1)
results_df['mean'] = (results_df['mean'] * 100).map('{:.2f}%'.format)
results_df['p_adj'] = results_df['p_adj'].map(lambda x: f'{x:.4f}' if pd.notna(x) else '---')

final_table = results_df[['Model', 'mean', '95% CI', 'p_adj']]
final_table = final_table.rename(columns={'mean': 'Mean Accuracy', 'p_adj': 'Adjusted p-value (vs. Baseline)'})
final_table = final_table.set_index('Model').reindex(['v_linear', 'v4', 'v6', 'v21'])

print(final_table)
print("\n--- Analysis complete. ---")

Loading all raw data sources...
Loaded X data: (34, 2476)
Loaded y labels: (34,)
Loaded PAC data: (34, 24)

Defining aggregation functions...
Building final n=34 datasets...
  X_linear shape: (34, 35)
  X_v4 shape:     (34, 37)
  X_v6 shape:     (34, 38)
  X_v21 shape:    (34, 62)

Defining final, fixed-parameter models...

Running 50 scores (5 splits, 10 repeats) on all n=34 subjects...
Cross-validation complete.

Calculating Bootstrap 95% Confidence Intervals...
Running paired hypothesis tests (Wilcoxon)...
Applying Bonferroni-Holm correction...

--- FINAL STATISTICAL RESULTS (n=34) ---
         Mean Accuracy             95% CI Adjusted p-value (vs. Baseline)
Model                                                                    
v_linear        64.57%  [59.57% - 69.71%]                             ---
v4              74.19%  [69.81% - 78.43%]                          0.0002
v6              71.29%  [67.33% - 75.00%]                          0.0067
v21             70.57%  [66.05% - 

In [6]:
### =========================================================================
### BLOCK 1: IMPORTS & LOADING (COMBINED)
### =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import re 

# --- Preprocessing & Pipeline ---
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
# *** NOTE: ClassificationReport and ConfusionMatrixDisplay are removed, as there is no test set ***

# --- Classifier Models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# --- File paths ---
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv'

print(f"Loading data from {TRAIN_FILE} and {TEST_FILE}...")

# --- Load Training Data ---
try:
    train_df = pd.read_csv(TRAIN_FILE)
    y_train_text = train_df['label']
    X_train = train_df.drop(columns=['label', 'subject_id'], errors='ignore') # <-- Fix: Drop subject_id
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find training file: {TRAIN_FILE}")
    exit()

# --- Load Testing Data ---
try:
    test_df = pd.read_csv(TEST_FILE)
    y_test_text = test_df['label']
    X_test = test_df.drop(columns=['label', 'subject_id'], errors='ignore') 
except FileNotFoundError:
    print(f"FATAL ERROR: Cannot find testing file: {TEST_FILE}")
    exit()

# --- *** MODIFICATION: Combine all data *** ---
print("Combining all 34 subjects into one dataset...")

# 1. Combine feature dataframes
X_full_combined = pd.concat([X_train, X_test], ignore_index=True)

# 2. Combine labels
y_full_text = np.concatenate([y_train_text, y_test_text])

# 3. Encode the *full* set of labels
encoder = LabelEncoder()
y_full_encoded = encoder.fit_transform(y_full_text)
class_names = encoder.classes_

print(f"Full combined dataset shape: {X_full_combined.shape}")
print(f"Full combined labels shape: {y_full_encoded.shape}")
print(f"Target classes: {class_names}")
print("-" * 50)


### =========================================================================
### BLOCK 2: DEFINE DOMAIN KNOWLEDGE (BRAIN REGIONS)
### =========================================================================
print("\nDefining brain region mappings...")

# Get a list of all channels from the columns (e.g., 'sampen_alpha_Fp1')
all_columns = X_full_combined.columns # Use the combined df
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1:
        all_channels.add(parts[-1]) 
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}

print(f"Found {len(all_channels)} unique channel names in the data.")

# --- Define Standard Regions ---
REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
    'temporal': ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10'],
    'occipital': ['O1', 'Oz', 'O2', 'PO7', 'PO8']
}

# --- Create final region-to-channel mapping ---
final_regions = {}
for region_name, std_channels in REGIONS.items():
    found_channels = [ch for ch in std_channels if ch in all_channels]
    final_regions[region_name] = found_channels
    print(f"  - {region_name.title()}: Found {len(found_channels)} channels {found_channels}")

# Create a 'global' region mapping for all channels found
final_regions['global'] = list(all_channels)
print("-" * 50)


### =========================================================================
### BLOCK 3: DEFINE THE FEATURE AGGREGATION FUNCTION
### =========================================================================
def aggregate_features(df, region_map):
    """
    Takes a wide dataframe and aggregates it down to 
    a small, hypothesis-driven dataframe (p=35).
    """
    print(f"\nAggregating features for {len(df)} subjects...")
    
    # Store new features in a dictionary
    agg_features = {}
    
    # --- 1. Keep PRE-AGGREGATED Features (Asymmetry, GFP) ---
    print("  - Hypothesis 0: Keeping 7 pre-aggregated features...")
    pre_agg_cols = [
        'frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 
        'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma'
    ]
    for col in pre_agg_cols:
        if col in df.columns:
            agg_features[col] = df[col]
        else:
            print(f"    - Warning: Pre-aggregated column '{col}' not found. Filling with 0.")
            agg_features[col] = 0

    # --- 2. Helper function for Complexity/Entropy (e.g., 'hfd_alpha_Fp1') ---
    def get_region_avg(df, feature_prefix, band, region_name):
        channels = region_map.get(region_name, [])
        if not channels:
            return pd.Series(0, index=df.index) # Return a Series of zeros
        
        cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
        
        if not cols_to_avg:
             return pd.Series(0, index=df.index)
        
        return df[cols_to_avg].mean(axis=1)

    # --- 3. Helper function for SEF/Centroid (e.g., 'Fp1_Centroid') ---
    def get_region_avg_spec(df, feature_suffix, region_name):
        channels = region_map.get(region_name, [])
        if not channels:
            return pd.Series(0, index=df.index)
        
        cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
        
        if not cols_to_avg:
             return pd.Series(0, index=df.index)
        return df[cols_to_avg].mean(axis=1)

    # --- 4. Create NEW Aggregated Features ---
    print("  - Hypothesis 1 & 2: Creating 8 Complexity features (HFD, DFA)...")
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')

    print("  - Hypothesis 3: Creating 2 Entropy features (Permutation)...")
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')

    print("  - Hypothesis 4: Creating 2 Spectral Shape features (SEF, Centroid)...")
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')

    print("  - Hypothesis 5: Creating 16 Spectral Power features (Abs/Rel)...")
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')

    # --- 5. Convert to DataFrame ---
    final_agg_df = pd.DataFrame(agg_features)
    final_agg_df = final_agg_df.fillna(0) # Fill any NaNs from failed averages with 0
    
    print(f"  - Aggregation complete. New shape: {final_agg_df.shape}")
    return final_agg_df
print("-" * 50)


### =========================================================================
### BLOCK 4: APPLY AGGREGATION TO FULL DATASET
### =========================================================================
print("\nApplying aggregation to combined data...")

# --- Apply the aggregation to the combined dataframe ---
X_full_agg = aggregate_features(X_full_combined, final_regions)

print(f"\nFinal aggregated features shape: {X_full_agg.shape}")
print(f"Final aggregated labels shape: {y_full_encoded.shape}")
print("-" * 50)


### =========================================================================
### BLOCK 5: DEFINE NEW, SIMPLER ML SEARCH SPACE
### =========================================================================
def get_simple_model_search_space():
    """
    Returns a search space for a simple (Scaler + Classifier) pipeline.
    We use class_weight='balanced' to handle the unbalanced data.
    """
    
    # --- Base pipeline (NO PCA, NO SELECTION) ---
    pipeline_simple = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', 'passthrough')
    ])

    print("\nModel search space defined (Simple Scaler + Classifier).")
    
    search_space = {
        
        # --- RandomForest ---
        'RF_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [RandomForestClassifier(random_state=42, class_weight='balanced')],
                'classifier__n_estimators': [50, 100, 200],
                'classifier__max_depth': [2, 3, 5],
                'classifier__min_samples_leaf': [3, 5]
            }
        },
        
        # --- SVM ---
        'SVM_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [SVC(probability=True, random_state=42, class_weight='balanced')],
                'classifier__C': [0.01, 0.1, 1.0, 10.0],
                'classifier__gamma': [0.01, 0.1, 'scale']
            }
        },

        # --- Logistic Regression ---
        'LogReg_Simple': {
            'pipeline': pipeline_simple,
            'params': {
                'classifier': [LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, class_weight='balanced')],
                'classifier__penalty': ['l1', 'l2'],
                'classifier__C': [0.01, 0.1, 1.0, 10.0]
            }
        },
    }
    
    return search_space
print("-" * 50)


### =========================================================================
### BLOCK 6: RUN CROSS-VALIDATION ON *ALL* 34 SUBJECTS
### =========================================================================

# --- Get the new search space ---
simple_search_space = get_simple_model_search_space()

# --- Use RepeatedStratifiedKFold for stable CV ---
cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
print(f"\nUsing RepeatedStratifiedKFold (5 splits, 5 repeats) on all n={len(X_full_agg)} subjects.")

all_results = []

for model_name, config in simple_search_space.items():
    print(f"\n--- Running GridSearch for: {model_name} ---")
    
    grid_search = GridSearchCV(
        estimator=config['pipeline'], 
        param_grid=config['params'], 
        cv=cv_strategy, 
        scoring='accuracy',
        n_jobs=-1,
        verbose=0,
        refit=True # <-- Refit on the *entire* dataset
    )
    
    # *** MODIFICATION: Fit on the full dataset ***
    grid_search.fit(X_full_agg, y_full_encoded)
    
    print(f"  - Best Mean CV Score: {grid_search.best_score_ * 100:.2f}%")
    all_results.append({
        'model_name': model_name,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    })

# --- Sort and show CV results ---
all_results.sort(key=lambda x: x['best_score'], reverse=True)
print("\n" + "="*40)
print("--- AGGREGATED MODEL CV COMPARISON (ON ALL 34 SUBJECTS) ---")
print("="*40)
for result in all_results:
    print(f"Model: {result['model_name']}")
    print(f"  Best Mean CV Accuracy: {result['best_score'] * 100:.2f}%")

# --- Get the BEST model (which is now fit on all 34 subjects) ---
best_overall_result = all_results[0]
best_pipeline = best_overall_result['best_estimator']

print(f"\n--- Winning Model & Hyperparameters ---")
print(f"Winner: {best_overall_result['model_name']}")
print(f"Winning Hyperparameters: {best_overall_result['best_params']}")
print("\n--- Model run complete. ---")


### =========================================================================
### BLOCK 7: ANALYZE FINAL FEATURE IMPORTANCE (FROM MODEL FIT ON ALL DATA)
### =========================================================================
print("\n" + "="*40)
print("--- FEATURE IMPORTANCE OF WINNING MODEL (FIT ON ALL 34) ---")
print("="*40)

# Get the final fitted classifier from the best pipeline
best_classifier = best_pipeline.named_steps['classifier']

# Get the feature names from our aggregated data
# *** MODIFICATION: Use X_full_agg ***
feature_names = X_full_agg.columns.tolist()

importances = None

# Method 1: For RandomForest
if isinstance(best_classifier, RandomForestClassifier):
    print("Model is RandomForest. Using .feature_importances_")
    importances = best_classifier.feature_importances_

# Method 2: For LogisticRegression (L1 or L2)
elif isinstance(best_classifier, LogisticRegression):
    print("Model is LogisticRegression. Using abs(.coef_[0])")
    importances = np.abs(best_classifier.coef_[0])

# Method 3: For SVM
elif isinstance(best_classifier, SVC):
    print("Model is SVM. Built-in importances are not reliable.")
    print("To get feature importance for SVM, you would need to run a")
    print("more complex analysis like SHAP or Permutation Importance.")

# --- Print the results ---
if importances is not None:
    # Create a DataFrame to view the results
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    
    # Sort by importance
    feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
    
    print("\n--- Top 10 Most Important Features ---")
    print(feature_importance_df.head(10))
    
    # Plot the results
    try:
        plt.figure(figsize=(10, 8))
        plt.title("Top 10 Feature Importances (Aggregated Model - All 34 Subjects)")
        plt.barh(
            feature_importance_df['feature'].head(10), 
            feature_importance_df['importance'].head(10)
        )
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.gca().invert_yaxis() # Show best feature at the top
        plt.tight_layout()
        plt.savefig('feature_importance_aggregated_all_subjects.png')
        print("\nSaved 'feature_importance_aggregated_all_subjects.png'")
        plt.close()
    except Exception as e:
        print(f"Could not plot feature importances: {e}")

else:
    print(f"Cannot get simple feature importances for model type: {type(best_classifier)}")

print("\n--- Full analysis complete. ---")

Loading data from /kaggle/working/master_dataset/train_UNBALANCED_dataset.csv and /kaggle/working/master_dataset/test_unbalanced_dataset.csv...
Combining all 34 subjects into one dataset...
Full combined dataset shape: (34, 2476)
Full combined labels shape: (34,)
Target classes: ['ASD' 'TD']
--------------------------------------------------

Defining brain region mappings...
Found 85 unique channel names in the data.
  - Frontal: Found 9 channels ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4']
  - Central: Found 7 channels ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4']
  - Parietal: Found 11 channels ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4']
  - Temporal: Found 6 channels ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10']
  - Occipital: Found 5 channels ['O1', 'Oz', 'O2', 'PO7', 'PO8']
--------------------------------------------------
--------------------------------------------------

Applying aggregation to combined data...

Aggregating features fo

In [7]:
import numpy as np
import pandas as pd
import warnings
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import bootstrap, wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# =========================================================================
# BLOCK 1: DATA LOADING
# =========================================================================
print("Loading all raw data sources...")

# --- File Paths ---
# !!! YOU MUST UPDATE THESE PATHS !!!
TRAIN_FILE = '/kaggle/working/master_dataset/train_UNBALANCED_dataset.csv'
TEST_FILE = '/kaggle/working/master_dataset/test_unbalanced_dataset.csv'
PAC_ASD_FILE = '/kaggle/input/pac-dataset/pac_features_ASD.csv'
PAC_TD_FILE = '/kaggle/input/pac-dataset/pac_features_TD.csv'

# --- 1. Load Wide Data (X) ---
try:
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)
    
    # Get labels before dropping
    y_train_text = train_df['label']
    y_test_text = test_df['label']

    # Drop labels/subject_id to align columns
    X_train_wide = train_df.drop(columns=['label', 'subject_id'], errors='ignore')
    X_test_wide = test_df.drop(columns=['label', 'subject_id'], errors='ignore')
    
    # Combine into one n=34 wide dataframe
    X_wide_full = pd.concat([X_train_wide, X_test_wide], ignore_index=True)
    
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find wide data files.")
    print(e)
    exit()

# --- 2. Load Labels (y) ---
y_full_text = np.concatenate([y_train_text, y_test_text])
encoder = LabelEncoder()
y = encoder.fit_transform(y_full_text)
print(f"Loaded X data: {X_wide_full.shape}")
print(f"Loaded y labels: {y.shape}")

# --- 3. Load PAC Data (for v21) ---
try:
    pac_df_asd = pd.read_csv(PAC_ASD_FILE)
    pac_df_td = pd.read_csv(PAC_TD_FILE)
    
    X_pac_agg = pd.concat([pac_df_asd, pac_df_td], ignore_index=True)
    # Drop metadata columns
    X_pac_agg = X_pac_agg.drop(columns=['subject_id', 'label'], errors='ignore')
    print(f"Loaded PAC data: {X_pac_agg.shape}")
    
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find PAC data files.")
    print(e)
    exit()

if len(X_wide_full) != len(X_pac_agg):
    print("FATAL ERROR: Row mismatch between wide data and PAC data.")
    exit()

# =========================================================================
# BLOCK 2: AGGREGATION FUNCTIONS & HELPERS
# =========================================================================
print("\nDefining aggregation functions...")

# --- 2a. Define Regions ---
all_columns = X_wide_full.columns
all_channels = set()
for col in all_columns:
    parts = col.split('_')
    if len(parts) > 1: all_channels.add(parts[-1])
all_channels.update([col.split('_')[0] for col in all_columns if '_' in col])
all_channels = {ch for ch in all_channels if ch.isalnum() and not ch.isdigit()}

REGIONS = {
    'frontal': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'AF3', 'AF4'],
    'central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4'],
    'parietal': ['CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO3', 'PO4'],
    'temporal': ['T7', 'T8', 'TP9', 'TP10', 'FT9', 'FT10'],
    'occipital': ['O1', 'Oz', 'O2', 'PO7', 'PO8']
}
final_regions = {}
for region_name, std_channels in REGIONS.items():
    final_regions[region_name] = [ch for ch in std_channels if ch in all_channels]
final_regions['global'] = list(all_channels)

# --- 2b. Helper Functions ---
def get_region_avg(df, feature_prefix, band, region_name):
    channels = final_regions.get(region_name, [])
    if not channels: return pd.Series(0, index=df.index)
    cols_to_avg = [f"{feature_prefix}_{band}_{ch}" for ch in channels if f"{feature_prefix}_{band}_{ch}" in df.columns]
    if not cols_to_avg: return pd.Series(0, index=df.index)
    return df[cols_to_avg].mean(axis=1)

def get_region_avg_spec(df, feature_suffix, region_name):
    channels = final_regions.get(region_name, [])
    if not channels: return pd.Series(0, index=df.index)
    cols_to_avg = [f"{ch}_{feature_suffix}" for ch in channels if f"{ch}_{feature_suffix}" in df.columns]
    if not cols_to_avg: return pd.Series(0, index=df.index)
    return df[cols_to_avg].mean(axis=1)

# --- 2c. Base 35-Feature Aggregation (v-Linear) ---
def build_agg_features(df):
    agg_features = {}
    # 1. Pre-aggregated (7)
    pre_agg_cols = ['frontal_asymmetry', 'mean_gfp', 'hemi_asym_delta', 'hemi_asym_theta', 'hemi_asym_alpha', 'hemi_asym_beta', 'hemi_asym_gamma']
    for col in pre_agg_cols:
        if col in df.columns: agg_features[col] = df[col]
        else: agg_features[col] = 0
    # 2. Complexity (8)
    agg_features['global_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'global')
    agg_features['global_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'global')
    agg_features['global_dfa_alpha'] = get_region_avg(df, 'dfa', 'alpha', 'global')
    agg_features['global_dfa_beta'] = get_region_avg(df, 'dfa', 'beta', 'global')
    agg_features['frontal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'frontal')
    agg_features['parietal_hfd_alpha'] = get_region_avg(df, 'hfd', 'alpha', 'parietal')
    agg_features['frontal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'frontal')
    agg_features['parietal_hfd_beta'] = get_region_avg(df, 'hfd', 'beta', 'parietal')
    # 3. Entropy (2)
    agg_features['global_permen_theta'] = get_region_avg(df, 'permen', 'theta', 'global')
    agg_features['global_permen_beta'] = get_region_avg(df, 'permen', 'beta', 'global')
    # 4. Spectral Shape (2)
    agg_features['global_centroid'] = get_region_avg_spec(df, 'Centroid', 'global')
    agg_features['global_sef95'] = get_region_avg_spec(df, 'SEF95', 'global')
    # 5. Spectral Power (16)
    power_types = ['rel_power', 'abs_power']
    bands = ['delta', 'theta', 'alpha', 'beta']
    for p_type in power_types:
        for band in bands:
            agg_features[f"global_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'global')
            agg_features[f"frontal_{p_type}_{band}"] = get_region_avg(df, p_type, band, 'frontal')
    
    return agg_features

# =========================================================================
# BLOCK 3: CREATE FINAL (n=34) DATASETS
# =========================================================================
print("Building final n=34 datasets...")

# --- 3a. Get Base 35 Features ---
agg_dict_base = build_agg_features(X_wide_full)
X_linear = pd.DataFrame(agg_dict_base).fillna(0)

# --- 3b. Create v4 (37 features) ---
agg_dict_v4 = agg_dict_base.copy()
agg_dict_v4['asymm_x_dfa'] = agg_dict_v4.get('frontal_asymmetry', 0) * agg_dict_v4.get('global_dfa_beta', 0)
num = agg_dict_v4.get('global_abs_power_delta', 0)
den = agg_dict_v4.get('global_abs_power_beta', 0) + 1e-6
agg_dict_v4['delta_beta_ratio'] = num / den
X_v4 = pd.DataFrame(agg_dict_v4).fillna(0)

# --- 3c. Create v6 (38 features) ---
agg_dict_v6 = agg_dict_base.copy()
num_tb = agg_dict_v6.get('global_abs_power_theta', 0)
den_tb = agg_dict_v6.get('global_abs_power_beta', 0) + 1e-6
agg_dict_v6['global_tbr'] = num_tb / den_tb
num_da = agg_dict_v6.get('global_abs_power_delta', 0)
den_da = agg_dict_v6.get('global_abs_power_alpha', 0) + 1e-6
agg_dict_v6['global_dar'] = num_da / den_da
agg_dict_v6['parietal_abs_power_alpha'] = get_region_avg(X_wide_full, 'abs_power', 'alpha', 'parietal')
num_fp = agg_dict_v6.get('frontal_abs_power_alpha', 0)
den_fp = agg_dict_v6.get('parietal_abs_power_alpha', 0) + 1e-6
agg_dict_v6['fp_alpha_ratio'] = num_fp / den_fp
X_v6 = pd.DataFrame(agg_dict_v6).fillna(0)
# Drop the helper column
if 'parietal_abs_power_alpha' in X_v6.columns:
    X_v6 = X_v6.drop(columns=['parietal_abs_power_alpha'])

# --- 3d. Create v21 (v6 + PAC) ---
X_v21 = pd.concat([X_v6, X_pac_agg], axis=1).fillna(0)

print(f"  X_linear shape: {X_linear.shape}")
print(f"  X_v4 shape:     {X_v4.shape}")
print(f"  X_v6 shape:     {X_v6.shape}")
print(f"  X_v21 shape:    {X_v21.shape}")

# =========================================================================
# BLOCK 4: DEFINE FINAL, FIXED MODELS
# =========================================================================
print("\nDefining final, fixed-parameter models...")

# --- model_linear (v-Linear) ---
# *** UPDATED WITH NEW HYPERPARAMETERS ***
# Winner: LogReg_Simple
# Params: {'classifier__C': 1.0, 'classifier__penalty': 'l2', ...}
model_linear = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', LogisticRegression(
        C=1.0,  # <-- UPDATED from 0.1
        penalty='l2',
        solver='liblinear',
        class_weight='balanced',
        max_iter=1000, # <-- ADDED
        random_state=42
    ))
])

# --- model_v4 (Invented) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 7, 'classifier__weights': 'uniform'}
model_v4 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=7,
        weights='uniform',
        n_jobs=-1
    ))
])

# --- model_v6 (Literature) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 5, 'classifier__weights': 'uniform'}
model_v6 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=5,
        weights='uniform',
        n_jobs=-1
    ))
])

# --- model_v21 (PAC) ---
# Winner: KNN_Simple
# Params: {'classifier__n_neighbors': 7, 'classifier__weights': 'uniform'}
model_v21 = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=7,
        weights='uniform',
        n_jobs=-1
    ))
])

# =========================================================================
# BLOCK 5: RUN STATISTICAL EVALUATION
# =========================================================================

# --- 5a. Run Repeated Cross-Validation ---
# We use n_splits=5, n_repeats=10 to match your paper's code
# Using 10 repeats to get 50 scores.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

print(f"\nRunning {cv.get_n_splits()} scores (5 splits, 10 repeats) on all n=34 subjects...")
all_scores = {}
all_scores['v_linear'] = cross_val_score(model_linear, X_linear, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v4'] = cross_val_score(model_v4, X_v4, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v6'] = cross_val_score(model_v6, X_v6, y, cv=cv, scoring='accuracy', n_jobs=-1)
all_scores['v21'] = cross_val_score(model_v21, X_v21, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("Cross-validation complete.\n")


# --- 5b. Calculate Bootstrap CIs ---
print("Calculating Bootstrap 95% Confidence Intervals...")
results = {}

for model_name, scores in all_scores.items():
    mean_accuracy = np.mean(scores)
    # (data,) must be a tuple for scipy.stats.bootstrap
    ci_result = bootstrap((scores,), np.mean, confidence_level=0.95, 
                          random_state=42, method='BCa')
    
    results[model_name] = {
        'mean': mean_accuracy,
        'ci_low': ci_result.confidence_interval.low,
        'ci_high': ci_result.confidence_interval.high
    }

# --- 5c. Run Paired Hypothesis Tests (Wilcoxon) ---
print("Running paired hypothesis tests (Wilcoxon)...")

# One-sided test: is the model *greater* than the baseline?
p_v4 = wilcoxon(all_scores['v4'], all_scores['v_linear'], alternative='greater').pvalue
p_v6 = wilcoxon(all_scores['v6'], all_scores['v_linear'], alternative='greater').pvalue
p_v21 = wilcoxon(all_scores['v21'], all_scores['v_linear'], alternative='greater').pvalue

raw_p_values = [p_v4, p_v6, p_v21]

# --- 5d. Apply Bonferroni-Holm Correction ---
print("Applying Bonferroni-Holm correction...\n")

reject, p_adjusted, _, _ = multipletests(raw_p_values, alpha=0.05, 
                                         method='holm')

# Store adjusted p-values
results['v4']['p_adj'] = p_adjusted[0]
results['v6']['p_adj'] = p_adjusted[1]
results['v21']['p_adj'] = p_adjusted[2]
results['v_linear']['p_adj'] = np.nan # Baseline has no p-value

# --- 5e. Print Final Results Table ---
print("--- FINAL STATISTICAL RESULTS (n=34) ---")

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.rename_axis('Model').reset_index()
results_df['95% CI'] = results_df.apply(lambda row: f"[{row['ci_low']*100:.2f}% - {row['ci_high']*100:.2f}%]", axis=1)
results_df['mean'] = (results_df['mean'] * 100).map('{:.2f}%'.format)
results_df['p_adj'] = results_df['p_adj'].map(lambda x: f'{x:.4f}' if pd.notna(x) else '---')

final_table = results_df[['Model', 'mean', '95% CI', 'p_adj']]
final_table = final_table.rename(columns={'mean': 'Mean Accuracy', 'p_adj': 'Adjusted p-value (vs. Baseline)'})
final_table = final_table.set_index('Model').reindex(['v_linear', 'v4', 'v6', 'v21'])

print(final_table)
print("\n--- Analysis complete. ---")

Loading all raw data sources...
Loaded X data: (34, 2476)
Loaded y labels: (34,)
Loaded PAC data: (34, 24)

Defining aggregation functions...
Building final n=34 datasets...
  X_linear shape: (34, 35)
  X_v4 shape:     (34, 37)
  X_v6 shape:     (34, 38)
  X_v21 shape:    (34, 62)

Defining final, fixed-parameter models...

Running 50 scores (5 splits, 10 repeats) on all n=34 subjects...
Cross-validation complete.

Calculating Bootstrap 95% Confidence Intervals...
Running paired hypothesis tests (Wilcoxon)...
Applying Bonferroni-Holm correction...

--- FINAL STATISTICAL RESULTS (n=34) ---
         Mean Accuracy             95% CI Adjusted p-value (vs. Baseline)
Model                                                                    
v_linear        66.62%  [62.33% - 71.19%]                             ---
v4              74.19%  [69.81% - 78.43%]                          0.0062
v6              71.29%  [67.33% - 75.00%]                          0.1107
v21             70.57%  [66.05% - 